# Non-paired backup notebook

Copies the source lakehouse Tables and Files folders to a secondary-region storage account and writes a manifest for deterministic restore.

In [ ]:
import json
from datetime import datetime, timezone

try:
    from notebookutils import fs as fabric_fs
except ImportError:
    try:
        from mssparkutils import fs as fabric_fs
    except ImportError:
        fabric_fs = None

if fabric_fs is None:
    raise RuntimeError('This notebook must run in Fabric or Synapse with notebookutils/mssparkutils.fs available.')

source_workspace_identifier = '__SOURCE_WORKSPACE_IDENTIFIER__'
source_lakehouse_name = '__SOURCE_LAKEHOUSE_NAME__'
backup_root = 'abfss://fabric-exports@storageaccount.dfs.core.windows.net/fabric-bcdr-demo'
warehouse_export_path = ''

backup_timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
source_root = f'abfss://{source_workspace_identifier}@onelake.dfs.fabric.microsoft.com/{source_lakehouse_name}.Lakehouse'
backup_run_root = f'{backup_root}/{source_workspace_identifier}/{source_lakehouse_name}/{backup_timestamp}'

print(f'Source root: {source_root}')
print(f'Backup root: {backup_run_root}')

In [ ]:
copy_plan = [
    {
        'name': 'tables',
        'source': f'{source_root}/Tables',
        'target': f'{backup_run_root}/lakehouse/Tables'
    },
    {
        'name': 'files',
        'source': f'{source_root}/Files',
        'target': f'{backup_run_root}/lakehouse/Files'
    }
]

if warehouse_export_path:
    copy_plan.append({
        'name': 'warehouse-export',
        'source': warehouse_export_path,
        'target': f'{backup_run_root}/warehouse-export'
    })

for entry in copy_plan:
    print(f"Copying {entry['name']} -> {entry['target']}")
    fabric_fs.cp(entry['source'], entry['target'], True)

print('Backup copy complete')

In [ ]:
manifest = {
    'backupTimestampUtc': backup_timestamp,
    'scenario': 'nonpaired',
    'source': {
        'workspaceIdentifier': source_workspace_identifier,
        'lakehouseName': source_lakehouse_name,
        'lakehouseRoot': source_root
    },
    'backup': {
        'root': backup_run_root,
        'entries': copy_plan
    }
}

manifest_path = f'{backup_run_root}/backup-manifest.json'
fabric_fs.put(manifest_path, json.dumps(manifest, indent=2), True)

print(json.dumps(manifest, indent=2))
print(f'Manifest written to: {manifest_path}')